# Step 2: Launch AutoGluon TimeSeries Training

Uses `ModelTrainer` (SageMaker SDK v3) with the AWS-managed AutoGluon DLC image to train an AutoGluon `TimeSeriesPredictor`.

## Configuration

In [ ]:
import boto3
import sagemaker

# --- Edit these as needed ---
REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-timeseries"
INSTANCE_TYPE = "ml.m5.2xlarge"
AG_VERSION = "1.5"
PY_VERSION = "py312"
JOB_NAME = "autogluon-timeseries"

S3_TRAIN = f"s3://{BUCKET}/{S3_PREFIX}/processed/train/"
S3_TEST = f"s3://{BUCKET}/{S3_PREFIX}/processed/test/"
S3_OUTPUT = f"s3://{BUCKET}/{S3_PREFIX}/model/"

# Hyperparameters
PREDICTION_LENGTH = 84
PRESETS = "medium_quality"
TIME_LIMIT = 3600
EVAL_METRIC = "MASE"

print(f"Region:    {REGION}")
print(f"S3 train:  {S3_TRAIN}")
print(f"S3 test:   {S3_TEST}")
print(f"S3 output: {S3_OUTPUT}")

## Discover IAM Role

In [ ]:
iam = boto3.client("iam")
role_arn = None
paginator = iam.get_paginator("list_roles")
for page in paginator.paginate():
    for role in page["Roles"]:
        if "SageMaker" in role["RoleName"] or "sagemaker" in role["RoleName"]:
            role_arn = role["Arn"]
            break
    if role_arn:
        break

# Uncomment to override:
# role_arn = "arn:aws:iam::123456789012:role/YourSageMakerRole"

assert role_arn, "No SageMaker IAM role found. Set role_arn manually above."
print(f"Using role: {role_arn}")

## Resolve Training Image

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session

session = Session()

image_uri = image_uris.retrieve(
    "autogluon", region=REGION, version=AG_VERSION,
    py_version=PY_VERSION, image_scope="training",
    instance_type=INSTANCE_TYPE,
)
print(f"Image URI: {image_uri}")

## Create ModelTrainer and Launch Training

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    Compute,
    SourceCode,
    OutputDataConfig,
    StoppingCondition,
)

hyperparameters = {
    "prediction-length": PREDICTION_LENGTH,
    "presets": PRESETS,
    "time-limit": TIME_LIMIT,
    "eval-metric": EVAL_METRIC,
    "target": "target",
    "id-column": "item_id",
    "timestamp-column": "timestamp",
}

input_data_config = [
    {
        "channel_name": "train",
        "data_source": {
            "s3_data_source": {
                "s3_uri": S3_TRAIN,
                "s3_data_type": "S3Prefix",
            }
        },
    },
    {
        "channel_name": "test",
        "data_source": {
            "s3_data_source": {
                "s3_uri": S3_TEST,
                "s3_data_type": "S3Prefix",
            }
        },
    },
]

trainer = ModelTrainer(
    training_image=image_uri,
    role=role_arn,
    source_code=SourceCode(
        source_dir=".",
        entry_script="train.py",
    ),
    compute=Compute(
        instance_type=INSTANCE_TYPE,
        instance_count=1,
        volume_size_in_gb=50,
        keep_alive_period_in_seconds=0,
    ),
    output_data_config=OutputDataConfig(
        s3_output_path=S3_OUTPUT,
    ),
    hyperparameters=hyperparameters,
    base_job_name=JOB_NAME,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    sagemaker_session=session,
)

training_job = trainer.train(
    input_data_config=input_data_config,
    wait=True,
    logs=True,
)

## Training Results

In [ ]:
job_name = training_job.name
print(f"Training complete: {job_name}")
print(f"Console: https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{job_name}")